<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/ShortTrades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install pandas-ta
!pip install ta

In [43]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import ta
import numpy as np
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress
#from transformers import pipeline
import time
print("Libraries Installed!")

0.2.66
Libraries Installed!


In [44]:


def most_recent_quarter_start(today=None):
    if today is None:
        today = pd.Timestamp.today().normalize()
    year = today.year
    month = today.month

    # Determine quarter start months: Jan, Apr, Jul, Oct
    if month >= 10:
        q_start = pd.Timestamp(year, 10, 1)
    elif month >= 7:
        q_start = pd.Timestamp(year, 7, 1)
    elif month >= 4:
        q_start = pd.Timestamp(year, 4, 1)
    else:
        q_start = pd.Timestamp(year, 1, 1)

    return q_start

# Example usage
print("Today:", pd.Timestamp.today().normalize())
print("Most recent quarter start:", most_recent_quarter_start())

Today: 2025-11-02 00:00:00
Most recent quarter start: 2025-10-01 00:00:00


In [45]:
def macdv(prices, fast=12, slow=26, signal=9, atr_window=10, thresholds=(50, 150)):
    """
    Compute MACD-V (volatility normalized MACD).

    Parameters
    ----------
    prices : pd.Series
        Price series (e.g. closing prices).
    fast : int
        Fast EMA period.
    slow : int
        Slow EMA period.
    signal : int
        Signal EMA period for MACD line.
    atr_window : int
        ATR lookback window.
    thresholds : tuple
        (lower, upper) thresholds for neutral/ranging and extreme momentum zones.

    Returns
    -------
    pd.DataFrame with columns:
        - MACDV : MACD-V value
        - Signal : EMA of MACDV
        - Histogram : MACDV - Signal
        - EntryFlag : True when momentum is strong enough, False otherwise
    """
    # --- Step 1: EMAs for MACD ---
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_raw = ema_fast - ema_slow

    # --- Step 2: ATR for normalization ---
    high = prices.shift(1) * (1 + 0.01)   # synthetic highs/lows if OHLC not available
    low = prices.shift(1) * (1 - 0.01)
    close = prices
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(atr_window).mean()

    # --- Step 3: Normalize MACD by ATR ---
    macdv = (macd_raw / atr) * 100

    # --- Step 4: Signal line and histogram ---
    signal_line = macdv.ewm(span=signal, adjust=False).mean()
    histogram = macdv - signal_line

    # --- Step 5: Entry conditions ---
    lower, upper = thresholds
    entry_flag = ((macdv.abs() > lower) & (macdv.abs() < upper)) | (macdv.abs() > upper)

    df = pd.DataFrame({
        "MACDV": macdv,
        "Signal": signal_line,
        "Histogram": histogram,
        "EntryFlag": entry_flag
    })
    curr = df.iloc[-1]

    return curr.EntryFlag

def money_flow_signals(df, period=10):
    """
    Calculate Money Flow Index (MFI) and generate signals:
    - Positive money flow (TP > TP_prev)
    - Divergence (Price vs MFI mismatch)

    Parameters:
        df (pd.DataFrame): DataFrame with columns ["High", "Low", "Close", "Volume"]
        period (int): Lookback period for MFI (default=10)

    Returns:
        pd.DataFrame with added columns: ["TypicalPrice", "MFI", "PositiveFlow", "Divergence"]
    """

    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Step 1: Typical Price
    df["TypicalPrice"] = (df["High"] + df["Low"] + df["Close"]) / 3

    # Step 2: Raw Money Flow
    df["RawMoneyFlow"] = df["TypicalPrice"] * df["Volume"]

    # Step 3: Positive & Negative Flow
    df["PositiveFlow"] = np.where(df["TypicalPrice"] > df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)
    df["NegativeFlow"] = np.where(df["TypicalPrice"] < df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)

    # Step 4: Money Flow Ratio & MFI
    pos_flow = df["PositiveFlow"].rolling(period).sum()
    neg_flow = df["NegativeFlow"].rolling(period).sum()
    money_flow_ratio = pos_flow / neg_flow.replace(0, np.nan)
    df["MFI"] = 100 - (100 / (1 + money_flow_ratio))

    # Step 5: Positive Money Flow Signal
    df["PositiveFlowSignal"] = df["TypicalPrice"] > df["TypicalPrice"].shift(1)

    # Step 6: Divergence Detection
    df["PriceHigh"] = df["Close"].rolling(period).max()
    df["PriceLow"] = df["Close"].rolling(period).min()
    df["MFIHigh"] = df["MFI"].rolling(period).max()
    df["MFILow"] = df["MFI"].rolling(period).min()

    def divergence(row):
        if np.isnan(row["MFI"]):
            return None
        # Price makes higher high, but MFI does not
        if row["Close"] >= row["PriceHigh"] and row["MFI"] < row["MFIHigh"]:
            return "Bearish Divergence ⚠️"
        # Price makes lower low, but MFI does not
        elif row["Close"] <= row["PriceLow"] and row["MFI"] > row["MFILow"]:
            return "Bullish Divergence ✅"
        else:
            return "No Divergence"

    df["Divergence"] = df.apply(divergence, axis=1)
    df['Entry_signal']= (df["MFI"] > 50) & (df["Divergence"] == "No Divergence")
    curr = df.iloc[-1]

    return curr.Entry_signal




In [46]:
# List of ETFs to analyze
recent_quarter = most_recent_quarter_start()
df_raw = pd.read_csv('short_list.csv')
etfs = df_raw['Asset'].to_list()
print(etfs)

print(len(etfs))

['FI', 'ARE', 'MOS', 'CSGP', 'KVUE', 'AJG', 'HRL', 'IP', 'FDS', 'PGR', 'SW', 'ICE', 'BRO', 'CHTR', 'BX', 'MO', 'CMCSA', 'TMUS', 'CPAY', 'PM', 'META', 'APD', 'ERIE', 'T', 'UDR', 'MMC', 'KKR', 'OKE', 'BALL', 'FIS', 'LIN', 'OXY', 'NWS', 'CTVA', 'CCI', 'STZ', 'AMT', 'LYB', 'CLX', 'MDLZ', 'HSY', 'MAA', 'EG', 'AVB', 'AMP', 'VICI', 'SYY', 'HBAN', 'ALL', 'CF', 'CAG', 'WY', 'APO', 'CPT', 'EQR', 'AMCR', 'GIS', 'EOG', 'SBAC', 'TAP', 'KHC', 'MTCH', 'GL', 'KR', 'TRGP', 'PPG', 'BITO', 'BETH', 'PSCC', 'USO', 'BJK', 'USL', 'BNO', 'GBTC', 'IBIT', 'BTC', 'XLP', 'VEGI', 'VDC', 'CUT', 'IYK', 'EINC', 'WOOD', 'XLRE', 'MOO', 'BPAY', 'XLB', 'IYR', 'TOLZ', 'BWZ', 'KXI', 'VNQ', 'MORT', 'BWX', 'IAK', 'EWG', 'VTIP', 'USFR', 'PVI', 'IBND', 'PICB', 'PULT', 'STIP', 'PSR', 'IAT', 'REZ', 'TIPX', 'VAW', 'CMBS', 'ISHG', 'VTC', 'IGOV', 'VFH', 'LQDB', 'IYE', 'XLF', 'XLE', 'PFUT', 'IAI', 'IYF', 'LQD', 'RSPA', 'VDE', 'PHO', 'AIVL', 'IYM', 'PDBA', 'IHAK', 'RWR', 'USRT', 'PSCF', 'VCLT', 'REET', 'SPLB', 'IFGL', 'GCC', 'IYG', '

In [47]:
# Filter ETFs or stocks for liquidity
def filter_by_liquidity(etf_df, ticker_col="Asset", min_dollar_vol=1e6, lookback_days=30):
    liquid_etfs = []

    for ticker in etf_df[ticker_col]:
        try:
            # Fetch daily historical data
            data = yf.download(ticker, period=f"{lookback_days*2}d", interval="1d", auto_adjust=True)

            if data.empty:
                continue

            # Calculate dollar volume (Close × Volume)
            data["dollar_volume"] = data["Close"] * data["Volume"]

            # Calculate rolling average over lookback_days
            avg_dollar_volume = data["dollar_volume"].rolling(window=lookback_days).mean().iloc[-1]

            # Check liquidity condition
            if avg_dollar_volume >= min_dollar_vol:
                liquid_etfs.append(ticker)

        except Exception as e:
            print(f"Error fetching {ticker}: {e}")

    # Return filtered DataFrame
    return etf_df[etf_df[ticker_col].isin(liquid_etfs)]

# Example usage
df = pd.DataFrame({"Assets": etfs})
liquid_df = filter_by_liquidity(df, ticker_col="Assets")
df_o = df_raw[df_raw['Asset'].isin(liquid_df['Assets'])]
etfs = df_o['Asset'].to_list()
print("")
print(etfs)
print(len(etfs))



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


['FI', 'ARE', 'MOS', 'CSGP', 'KVUE', 'AJG', 'HRL', 'IP', 'FDS', 'PGR', 'SW', 'ICE', 'BRO', 'CHTR', 'BX', 'MO', 'CMCSA', 'TMUS', 'CPAY', 'PM', 'META', 'APD', 'ERIE', 'T', 'UDR', 'MMC', 'KKR', 'OKE', 'BALL', 'FIS', 'LIN', 'OXY', 'NWS', 'CTVA', 'CCI', 'STZ', 'AMT', 'LYB', 'CLX', 'MDLZ', 'HSY', 'MAA', 'EG', 'AVB', 'AMP', 'VICI', 'SYY', 'HBAN', 'ALL', 'CF', 'CAG', 'WY', 'APO', 'CPT', 'EQR', 'AMCR', 'GIS', 'EOG', 'SBAC', 'TAP', 'KHC', 'MTCH', 'GL', 'KR', 'TRGP', 'PPG', 'BITO', 'USO', 'BNO', 'GBTC', 'IBIT', 'BTC', 'XLP', 'VDC', 'IYK', 'XLRE', 'MOO', 'XLB', 'IYR', 'BWZ', 'KXI', 'VNQ', 'MORT', 'BWX', 'IAK', 'EWG', 'VTIP', 'USFR', 'IBND', 'PICB', 'PULT', 'STIP', 'IAT', 'REZ', 'TIPX', 'VAW', 'CMBS', 'ISHG', 'VTC', 'IGOV', 'VFH', 'IYE', 'XLF', 'XLE', 'IAI', 'IYF', 'LQD', 'RSPA', 'VDE', 'PHO', 'IYM', 'PDBA', 'IHAK', 'RWR', 'USRT', 'VCLT', 'REET', 'SPLB', 'IYG', 'IGLB', 'EWA', 'LRN', 'LAZR', 'OEC', 'CRMT', 'SG', 'SEZL', 'CBRL', 'PLAY', 'ZSPC', 'COUR', 'BWIN', 'FLWS', 'PTLO', 'ATGE', 'SMPL', 'HAIN',

In [56]:
# Function to fetch historical weekly data
def anchored_vwap(ticker: str, lookback_weeks: int = 5):
    """
    Calculate the Anchored VWAP from the most recent high within the past `lookback_weeks`
    and create a 'Signal' column that gives 'Buy' when Close > Anchored_VWAP, else 'No-Buy'.

    Args:
        ticker (str): Stock ticker symbol
        lookback_weeks (int): Number of weeks to look back for the highest price

    Returns:
        pandas.DataFrame: DataFrame with OHLC, Volume, Anchored_VWAP, and Signal columns
    """
    try:
        # --- Fetch 6 months of daily data to cover the lookback window
        data = yf.download(ticker, period="3mo", interval="1d", progress=False,auto_adjust=True)
        if isinstance(data.columns, pd.MultiIndex):
          data.columns = data.columns.get_level_values(0)  # keep only first level

        if data.empty:
            raise ValueError(f"No data retrieved for ticker {ticker}")

        data.dropna(inplace=True)
        data.index = pd.to_datetime(data.index)

        # --- Ensure we have enough data for the lookback period
        min_days_required = lookback_weeks * 5  # ~5 trading days per week
        if len(data) < min_days_required:
            raise ValueError(f"Insufficient data: {len(data)} days available, {min_days_required} required")

        # --- Find the most recent high within the past lookback_weeks
        recent_period = data.tail(min_days_required)
        anchor_date = recent_period['Low'].idxmin()

        # --- Verify anchor_date is a valid Timestamp
        if not isinstance(anchor_date, pd.Timestamp):
            raise ValueError(f"Invalid anchor_date: {anchor_date}. Expected a Timestamp.")

        anchor_price = recent_period.loc[anchor_date, 'Low']

        # --- Use .date() safely since we confirmed anchor_date is a Timestamp
        print(f"Anchored VWAP for {ticker} starting from {anchor_date.date()} (recent low = {anchor_price:.2f})")

        # --- Slice data from the anchor date onwards
        anchor_data = data.loc[anchor_date:]

        # --- Compute VWAP starting from the anchor date
        # Use typical price ((H+L+C)/3) for more accurate VWAP
        typical_price = (anchor_data['High'] + anchor_data['Low'] + anchor_data['Close']) / 3
        q = anchor_data['Volume']
        pv = (typical_price * q).cumsum()
        v = q.cumsum()

        # --- Avoid division by zero
        avwap = pv / v.where(v != 0, np.nan)

        # --- Add Anchored VWAP to the full dataset
        data['Anchored_VWAP'] = np.nan  # Initialize with NaN
        data.loc[anchor_date:, 'Anchored_VWAP'] = avwap

        # --- Create Buy/No-Buy Signal
        # Only apply signal where Anchored_VWAP is not NaN
        data['Signal'] = np.where(
            (data['Close'] < data['Anchored_VWAP']) & (data['Anchored_VWAP'].notna()),
            True,
            False
        )

        return data[['Anchored_VWAP', 'Signal']]

    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")
        return None

def get_monthly_data(ticker):
  try:
      df = yf.download(ticker, period="10y", interval="1mo",auto_adjust=True)

      df['10_month_SMA'] = df['Close'].rolling(window=10).mean()
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
      df['RVOL'] = df['Volume'] / df['Volume'].rolling(window=10).mean()
      df['RVOL_Slope'] = df['RVOL'].diff()
      # Calculate ADX, +DMI and -DMI
      high = df['High']
      low = df['Low']
      close = df['Close']
      # Calculate directional movements
      up_move = high.diff()
      down_move = -low.diff()
      plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
      minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
      # Calculate True Range (TR)
      tr1 = high - low
      tr2 = (high - close.shift()).abs()
      tr3 = (low - close.shift()).abs()
      tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
      # Smooth TR, +DM, and -DM using Wilder’s smoothing
      atr = tr.rolling(window=10).sum()
      plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
      minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
      plus_dm_smoothed = plus_dm_series.rolling(window=10).sum()
      minus_dm_smoothed = minus_dm_series.rolling(window=10).sum()
      # Directional Indicators
      plus_di = 100 * (plus_dm_smoothed / atr)
      minus_di = 100 * (minus_dm_smoothed / atr)
      # DX and ADX
      dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
      adx = dx.rolling(window=10).mean()
      # Add results to original DataFrame
      df['+DI'] = plus_di
      df['-DI'] = minus_di
      df['ADX'] = adx
      df['di_flag'] = df['-DI'] > df['+DI']
      df['di_flag'] = df['di_flag'].astype(int)
      df['adx_indicator'] = np.where(df['ADX'] > 25, 1, 0)
      df['adx_signal'] = df['adx_indicator'] * df['di_flag']

      return df
  except Exception as e:
      print("There is an error getting monthly data", e)

def get_weekly_data(ticker):
  try:
      df = yf.download(ticker, period="2y", interval="1wk",auto_adjust=True)

      df['10_week_SMA'] = df['Close'].rolling(window=10).mean()
      df['30_week_SMA'] = df['Close'].rolling(window=30).mean()
      df['ATR'] = compute_atr(df, 10)
      df['OBV'] = compute_obv(df)
      df['OBV_Slope'] = df['OBV'].diff()
      df['30_week_avg_volume'] = df['Volume'].rolling(window=30).mean()
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']

      # Compute raw EFI
      df['EFI'] = (df['Close'].diff()) * df['Volume']
      # Compute EMA of EFI
      df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
      # Determine if EFI_EMA is rising or falling
      df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
      # Calculate ADX, +DMI and -DMI
      high = df['High']
      low = df['Low']
      close = df['Close']
      # Calculate directional movements
      up_move = high.diff()
      down_move = -low.diff()
      plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
      minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
      # Calculate True Range (TR)
      tr1 = high - low
      tr2 = (high - close.shift()).abs()
      tr3 = (low - close.shift()).abs()
      tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
      # Smooth TR, +DM, and -DM using Wilder’s smoothing
      atr = tr.rolling(window=10).sum()
      plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
      minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
      plus_dm_smoothed = plus_dm_series.rolling(window=10).sum()
      minus_dm_smoothed = minus_dm_series.rolling(window=10).sum()
      # Directional Indicators
      plus_di = 100 * (plus_dm_smoothed / atr)
      minus_di = 100 * (minus_dm_smoothed / atr)
      # DX and ADX
      dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
      adx = dx.rolling(window=10).mean()
      # Add results to original DataFrame
      df['+DI'] = plus_di
      df['-DI'] = minus_di
      df['ADX'] = adx
      df['di_flag'] = df['-DI'] > df['+DI']
      df['di_flag'] = df['di_flag'].astype(int)
      df['adx_indicator'] = np.where(df['ADX'] > 25, 1, 0)
      df['adx_signal'] = df['adx_indicator'] * df['di_flag']

      # --- Overhead Resistance Filter ---
      recent_52_weeks = df[-52:]
      min_close_52w = recent_52_weeks['Close'].min()
      last_close = df['Close'].iloc[-1].iloc[0]
      # --- Above 52 weeks Low ---
      df['above_52w_low'] = last_close > min_close_52w
      df['below_52w_low'] = last_close < min_close_52w

      return df
  except Exception as e:
      print("There is an error getting weekly data", e)

def is_macd_bullish(df):
    """
    Determines if there is a bullish signal on the MACD indicator.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] < curr['Signal_Line'].iloc[-1]
      below_zero_line = curr['MACD_Line'].iloc[-1] < 0


      return macd_crossover, below_zero_line

    except Exception as e:
      print("Something went wrong while computing the MACD", e)

def calculate_vwap(df):
  """ Calculates vwap"""
  try:
    Typical_Price = (df['Close'].values + df['High'].values + df['Low'].values) / 3
    TPV = Typical_Price * df['Volume'].values
    vwap = TPV.cumsum() / df['Volume'].values.cumsum()
    return vwap
  except Exception as e:
    print("Something went wrong while computing the VWAP:", e)
    return None


def calculate_ma(data, length=10, ma_type="WMA"):
    if ma_type == "SMA":
        return data.rolling(window=length).mean()
    elif ma_type == "EMA":
        return data.ewm(span=length, adjust=False).mean()
    elif ma_type == "WMA":
        weights = np.arange(1, length+1)
        return data.rolling(length).apply(lambda x: np.dot(x, weights)/weights.sum(), raw=True)
    elif ma_type == "VWMA":
        return ta.volume_weighted_average_price(data, length)


# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)


# Function to compute On-Balance Volume (OBV)
def compute_obv(df):
  try:
    # Calculate daily price change: 1 if price is up, -1 if down, 0 if unchanged
    price_change = df['Close'].diff()

    # Use price change to decide whether to add or subtract volume
    obv = (price_change > 0).astype(int) * df['Volume']  # Volume when price goes up
    obv -= (price_change < 0).astype(int) * df['Volume']  # Volume when price goes down

    # We accumulate the OBV by taking the cumulative sum of the volume changes
    obv = obv.cumsum()

    return obv
  except Exception as e:
      print("Something went wrong while computing the OBV", e)


# Function to calculate risk-reward ratio
def calculate_risk_reward(df):
  try:
    if df.empty or len(df) < 20:  # Ensure there are enough data points
        return np.nan

    latest_price = df['Close'].iloc[-1].iloc[0]

    # Use the ATR for setting support level
    atr = df['ATR'].iloc[-1]  # Latest ATR value
    price_ema = df['8_day_EMA'].iloc[-1]
    atr_multiple = 1.5 # You can adjust this multiplier based on your strategy

    # Calculate the support level using the ATR
    trailing = atr * atr_multiple
    stop      = price_ema + trailing

    return trailing, stop
  except Exception as e:
      print("Something went wrong while computing the reward-risk ratio", e)

# Function to fetch daily data
def get_daily_data(ticker):
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    #if isinstance(df.columns, pd.MultiIndex):
        #df.columns = df.columns.get_level_values(0)  # keep only first level
    df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
    df['50_day_avg_volume'] = df['Volume'].rolling(window=50).mean()
    df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
    df['15_day_EMA'] = df['Close'].ewm(span=15, adjust=False).mean()
    df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
    df['100_day_SMA'] = df['Close'].rolling(window=100).mean()
    df['200_day_SMA'] = df['Close'].rolling(window=200).mean()
    df['ATR'] = compute_atr(df, 10)
    df["8EMA_plus_ATR"] = df["8_day_EMA"] + (1.5* df["ATR"])
    df["8EMA_minus_ATR"] = df["8_day_EMA"] - (1.5* df["ATR"])
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
    # Calculate ADX, +DMI and -DMI
    high = df['High']
    low = df['Low']
    close = df['Close']
    # Calculate directional movements
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    # Calculate True Range (TR)
    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    # Smooth TR, +DM, and -DM using Wilder’s smoothing
    atr = tr.rolling(window=10).sum()
    plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
    minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
    plus_dm_smoothed = plus_dm_series.rolling(window=10).sum()
    minus_dm_smoothed = minus_dm_series.rolling(window=10).sum()
    # Directional Indicators
    plus_di = 100 * (plus_dm_smoothed / atr)
    minus_di = 100 * (minus_dm_smoothed / atr)
    # DX and ADX
    dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=10).mean()
    # Add results to original DataFrame
    df['+DI'] = plus_di
    df['-DI'] = minus_di
    df['ADX'] = adx
    df['di_flag'] = df['-DI'] > df['+DI']
    df['di_flag'] = df['di_flag'].astype(int)
    df['adx_indicator'] = np.where(df['ADX'] > 25, 1, 0)
    df['adx_signal'] = df['adx_indicator'] * df['di_flag']
    return df

# Function to fetch hourly data
def get_30mins_data(ticker):
    df = yf.download(ticker, interval='30m', period='60d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level

    df['8d_SMA'] = df['Close'].rolling(window=8).mean() # changed from 104
    # Short-term & medium-term moving averages
    df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['50_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['200_EMA'] = df['Close'].ewm(span=200, adjust=False).mean()
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']

    return df

def get_heikin_ashi_signal(ticker="AAPL", period="6mo", interval="1d"):
    # Fetch OHLC data
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Compute Heikin Ashi candles
    ha_df = pd.DataFrame(index=df.index)
    ha_df['HA_Close'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

    ha_open = []
    for i in range(len(df)):
        if i == 0:
             ha_open.append((df['Open'].iloc[i] + df['Close'].iloc[i]) / 2)
        else:
            ha_open.append((ha_open[i-1] + ha_df['HA_Close'].iloc[i-1]) / 2)
    ha_df['HA_Open'] = ha_open
    ha_df['HA_High'] = ha_df[['HA_Open', 'HA_Close']].assign(High=df['High']).max(axis=1)
    ha_df['HA_Low'] = ha_df[['HA_Open', 'HA_Close']].assign(Low=df['Low']).min(axis=1)

    # Combine with original
    df = df.join(ha_df)
    # Check for green candle with flat bottom
    last = df.iloc[-1]
    red_candle = last['HA_Close'] < last['HA_Open']
    flat_top = abs(last['HA_High'] - last['HA_Open']) < 0.001 * last['HA_Open']
    signal = red_candle and flat_top

    print(f"\n🔍 Checking {ticker} ({interval} timeframe)")
    print(f"HA_Open: {last['HA_Open']:.2f}, HA_Close: {last['HA_Close']:.2f}, HA_Low: {last['HA_Low']:.2f}")
    if signal:
        print("✅ Heikin Ashi candle is Red with a FLAT top — strong bearish signal!")
    elif red_candle:
        print("🟢 Candle is red but not flat-topped — still bearish, but less strong.")
    else:
        print("🔴 Not a bearish candle — no entry confirmation yet.")

    return signal, red_candle

# Function to check monthly trend
def is_monthly_trend_bearish(df):
    if df.empty:
        return False

    adx_ok    = df['adx_signal'].iloc[-1] == 1
    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['10_month_SMA'].iloc[-1]
    macd_bearish_signal, below_zero_line = is_macd_bullish(df)
    below_10_month_SMA = latest_price < latest_sma
    return below_10_month_SMA and adx_ok and macd_bearish_signal


# Function to check weekly trend
def is_weekly_trend_bearish(df2):
    if df2.empty:
        return False

    df = df2.copy()
    latest_price  = df['Close'].iloc[-1].iloc[0]
    latest_10sma  = df['10_week_SMA'].iloc[-1]
    below_10w_SMA = latest_price < latest_10sma
    latest_30sma  = df['30_week_SMA'].iloc[-1]
    below_30w_SMA = latest_price < latest_30sma
    macd_bearish_signal,below_zero_line = is_macd_bullish(df)
    adx_ok        = df['adx_signal'].iloc[-1] == 1
    trend_ok      = below_10w_SMA and below_30w_SMA and adx_ok
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Falling'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] < 0

    time.sleep(2)  # Add a delay of 1 second between requests

    return  trend_ok and macd_bearish_signal and elderforce_trend_ok and elderforce_ema_ok
             #and below_52w_low and elderforce_trend_ok and elderforce_ema_ok


# Function to check daily entry signal
def is_daily_entry_bearish(df2):
    if df2.empty:
        return False

    df = df2.copy()
    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_50sma = df['50_day_SMA'].iloc[-1]
    latest_100sma = df['100_day_SMA'].iloc[-1]
    latest_200sma = df['200_day_SMA'].iloc[-1]
    below_50sma = latest_price < latest_50sma
    below_100sma = latest_price < latest_100sma
    below_200sma = latest_price < latest_200sma
    is_50sma_below_100sma = latest_50sma < latest_100sma
    is_100sma_below_200sma = latest_100sma < latest_200sma
    macd_bearish_signal, below_zero_line = is_macd_bullish(df)
    adx_ok = df['adx_signal'].iloc[-1] == 1
    moving_averages_ok = below_50sma and below_100sma and is_50sma_below_100sma \
                         and below_200sma and is_100sma_below_200sma


    # Look for a breakout above 20-day SMA & RSI > 50
    return moving_averages_ok and adx_ok and macd_bearish_signal

# Check entry conditions
def check_entry_conditions(tickers):
    results = []
    for ticker in tickers:
      df = get_daily_data(ticker)
      latest_price        = df['Close'].iloc[-1].iloc[0]
      latest_sma          = df['50_day_SMA'].iloc[-1]
      latest_price_8ema   =  df['8_day_EMA'].iloc[-1]
      plus_8atr           = df['8EMA_plus_ATR'].iloc[-1]
      minus_8atr          = df['8EMA_minus_ATR'].iloc[-1]
      mfi_signal          = money_flow_signals(df)
      # Print results
      print(f"\nMoney Outflow indicator for {ticker} is:")
      print(not(mfi_signal))

      df_entry             = get_30mins_data(ticker)
      latest_priceh_8sma   = df_entry['8d_SMA'].iloc[-1]
      latest_priceh_21ema  = df_entry['21_EMA'].iloc[-1]
      latest_50ema         = df_entry['50_EMA'].iloc[-1]
      latest_200ema         = df_entry['200_EMA'].iloc[-1]
      latest_priceh        = df_entry['Close'].iloc[-1] #.iloc[0]
      macdHist_pos_hr      = df_entry['MACD_Hist'].iloc[-1] < 0
      HA_sell_signal_h,rc_h= get_heikin_ashi_signal(ticker, period="60d", interval="30m")


      refined_entry_signal = (latest_priceh <  latest_50ema) and (latest_50ema < latest_200ema) \
                             and HA_sell_signal_h


      if latest_price < minus_8atr :
        entry_signal = "Extended Short Entry"
      elif (latest_price < plus_8atr) and (latest_price >= minus_8atr): # and refined_entry_signal:
        entry_signal = "Aline Short Entry"
      else:
        entry_signal = "Skip"
      results.append([ticker, entry_signal])
    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results

# Multi-timeframe strategy check returning a DataFrame
def check_mtf_entry(tickers):
    results = []

    for ticker in tickers:
        monthly_df = get_monthly_data(ticker)
        weekly_df = get_weekly_data(ticker)
        daily_df = get_daily_data(ticker)

        if is_monthly_trend_bearish(monthly_df) and is_weekly_trend_bearish(weekly_df):
            if  is_daily_entry_bearish(daily_df):
                entry_signal = "Bearish Entry Confirmed ✅"
            else:
                entry_signal = "No Bearish Entry Yet on Daily Timeframe ⏳"
        else:
            entry_signal = "Monthly/Weekly  Trend is not Bearishh ❌"

        results.append([ticker, entry_signal])
        time.sleep(2)  # Add a delay of 1 second between requests

    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results



In [49]:
# Multi-time frame entry Check
etfs_to_check = df_o['Asset'].tolist()

df_signals = check_mtf_entry(etfs_to_check)

df_final = df_signals[df_signals['Entry_Signal'] =="Bearish Entry Confirmed ✅"]

df_final.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Asset,Entry_Signal
0,FI,Bearish Entry Confirmed ✅
5,AJG,Bearish Entry Confirmed ✅
6,HRL,Bearish Entry Confirmed ✅
7,IP,Bearish Entry Confirmed ✅
9,PGR,Bearish Entry Confirmed ✅


## Generate Sell list

In [57]:
#df_final = df_signals[df_signals['Entry_Signal'] =="Bearish Entry Confirmed ✅"]
final_etfs_to_check = df_final['Asset'].tolist()


sell_list = check_entry_conditions(final_etfs_to_check)


sell_list= sell_list[sell_list['Entry_Signal'].isin(['Aline Short Entry'])]

sell_list

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Money Outflow indicator for FI is:
True



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking FI (30m timeframe)
HA_Open: 66.40, HA_Close: 66.58, HA_Low: 66.32
🔴 Not a bearish candle — no entry confirmation yet.


[*********************100%***********************]  1 of 1 completed

...

[*********************100%***********************]  1 of 1 completed


Money Outflow indicator for SIGI is:
True



[*********************100%***********************]  1 of 1 completed


🔍 Checking SIGI (30m timeframe)
HA_Open: 74.75, HA_Close: 75.19, HA_Low: 74.75
🔴 Not a bearish candle — no entry confirmation yet.


,Asset,Entry_Signal
6,CHTR,Aline Short Entry
7,CMCSA,Aline Short Entry
8,TMUS,Aline Short Entry
9,CPAY,Aline Short Entry
11,UDR,Aline Short Entry
13,STZ,Aline Short Entry
14,MAA,Aline Short Entry
16,WY,Aline Short Entry
17,CPT,Aline Short Entry
18,EQR,Aline Short Entry


# Find and filter correlated assets to reduce concentration risk.

In [58]:

def get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66, period="3mo", interval="1d"):
    """
    Filters a ranked list of tickers to return only uncorrelated picks.

    Parameters:
    -----------
    tickers : list
        All candidate tickers.
    ranked_picks : list
        Ranked list of tickers (best to worst).
    threshold : float
        Correlation threshold (default 0.66).
    period : str
        Data period for yfinance (default "3mo").
    interval : str
        Data interval (default "1d").

    Returns:
    --------
    final_selection : list
        List of uncorrelated tickers.
    corr_matrix : DataFrame
        Correlation matrix of daily returns.
    """
    # Step 1: Get prices
    data = yf.download(tickers, period=period, interval=interval,auto_adjust=True)["Close"]
    data = data.ffill()

    # Step 2: Convert to daily returns
    returns = data.pct_change().dropna()

    # Step 3: Correlation matrix
    corr_matrix = returns.corr()

    # Step 4: Filter uncorrelated picks
    final_selection = []
    for pick in ranked_picks:
        if all(abs(corr_matrix.loc[pick, sel]) <= threshold for sel in final_selection):
            final_selection.append(pick)

    return final_selection, corr_matrix


# Example usage
tickers = sell_list['Asset'].tolist()  # Replace with your list of tickers
ranked_picks = tickers

final_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

#print("Final uncorrelated picks:", final_selection)
print("\nCorrelation matrix:\n", corr_matrix)

# Keep only rows where Asset is in filtered
#filtered_list = sell_list[sell_list["Asset"].isin(final_selection)]

#sell_list = filtered_list.copy()
sell_list

[*********************100%***********************]  24 of 24 completed


Correlation matrix:
 Ticker      BWIN       CAL       CBT      CHTR       CLW     CMCSA      CPAY  \
Ticker                                                                         
BWIN    1.000000  0.119500  0.260535  0.277304  0.219725  0.316939  0.275145   
CAL     0.119500  1.000000  0.381437  0.424659  0.402730  0.481118  0.494754   
CBT     0.260535  0.381437  1.000000  0.330515  0.505973  0.384083  0.366473   
CHTR    0.277304  0.424659  0.330515  1.000000  0.281579  0.762564  0.503553   
CLW     0.219725  0.402730  0.505973  0.281579  1.000000  0.281229  0.337160   
CMCSA   0.316939  0.481118  0.384083  0.762564  0.281229  1.000000  0.654528   
CPAY    0.275145  0.494754  0.366473  0.503553  0.337160  0.654528  1.000000   
CPT     0.207444  0.130536  0.490406  0.132671  0.467691  0.221869  0.303832   
EQR     0.162669  0.169029  0.393038  0.118734  0.413014  0.194192  0.301790   
HNST    0.087563  0.340865  0.251875  0.237888  0.161876  0.286295  0.521491   
INBK    0.206870  

,Asset,Entry_Signal
6,CHTR,Aline Short Entry
7,CMCSA,Aline Short Entry
8,TMUS,Aline Short Entry
9,CPAY,Aline Short Entry
11,UDR,Aline Short Entry
13,STZ,Aline Short Entry
14,MAA,Aline Short Entry
16,WY,Aline Short Entry
17,CPT,Aline Short Entry
18,EQR,Aline Short Entry


In [59]:
# Apply TA filters and prioritize ETFs
results = []
sell_list = sell_list[sell_list['Entry_Signal'].isin(['Aline Short Entry'])]


for etf in sell_list['Asset'].to_list():
   df =get_daily_data(etf)
   price = df['Close'].iloc[-1].iloc[0]
   below_50sma = price  < df['50_day_SMA'].iloc[-1]
   vwap_df     = anchored_vwap(etf, lookback_weeks=2)
   vwap        = vwap_df['Anchored_VWAP'].iloc[-1]
   vwap_signal = vwap_df['Signal'].iloc[-1]
   below_vwap  = price < vwap


   if below_50sma: # and below_vwap :
    trail, stop = calculate_risk_reward(df)
    entry_price = price - max(0., 0.05*trail)
    risk = np.abs(stop - entry_price)
    take_profit = entry_price - (2*risk)
    reward =  entry_price - take_profit
    support_level = stop
    risk_reward_ratio = reward / risk
    # Ensure risk is greater than zero before division
    if risk > 0:

        rr_ratio  = reward / risk
    else:
        rr_ratio = np.nan

    stop_loss_perc = ((entry_price-support_level )/entry_price )*100
    take_profit_perc = (( entry_price - take_profit)/entry_price )*100
    # Fetch the Entry_Signal from buy_list
    entry_signal = sell_list.loc[sell_list['Asset'] == etf, 'Entry_Signal'].values[0]

    # Append results with Entry_Signal
    results.append({
            "Asset": etf,
            "Risk-Reward": rr_ratio,
            "Stop Loss": support_level,
            "Take Profit": take_profit,
            "Current Price": price,
            "Entry Price": entry_price,
            "Trail Price": trail,
            "Entry Signal": entry_signal,  # Add entry signal
            "stop_loss_perc": stop_loss_perc,
            "take_profit_perc": take_profit_perc,
            "Anchored VWAP": vwap
        })

    time.sleep(2)  # Add a delay of 1 second between requests


# Sort ETFs by highest risk-to-reward ratio
try:
   df_results = pd.DataFrame(results).dropna().sort_values(by="Risk-Reward", ascending=True).reset_index(drop = True)
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  df_results = pd.DataFrame({"Asset": ["No Asset available"]})

df2 = df_results.merge(df_o[['Asset','Type', 'score']], on='Asset', how='left')
df2['timestamp'] = datetime.now()
df2 = df2.sort_values(by='score', ascending=True)
df2.head()

[*********************100%***********************]  1 of 1 completed


Anchored VWAP for CHTR starting from 2025-10-31 (recent low = 215.93)


[*********************100%***********************]  1 of 1 completed


Anchored VWAP for CMCSA starting from 2025-10-30 (recent low = 25.75)


[*********************100%***********************]  1 of 1 completed


...

[*********************100%***********************]  1 of 1 completed


Anchored VWAP for CBT starting from 2025-10-31 (recent low = 65.59)


[*********************100%***********************]  1 of 1 completed


Anchored VWAP for SIGI starting from 2025-10-30 (recent low = 74.20)


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,Type,score,timestamp
0,CHTR,2.0,254.720382,190.123975,233.839996,233.188246,13.035001,Aline Short Entry,-9.233800,18.467599,228.129995,Stock,-1.62,2025-11-02 22:14:23.464595
1,CMCSA,2.0,29.825525,23.686251,27.840000,27.779100,1.218000,Aline Short Entry,-7.366777,14.733554,27.195550,Stock,-1.44,2025-11-02 22:14:23.464595
2,TMUS,2.0,226.467020,175.774844,210.050003,209.569628,9.607496,Aline Short Entry,-8.062901,16.125802,209.846670,Stock,-1.44,2025-11-02 22:14:23.464595
3,CPAY,2.0,285.229650,208.615444,260.350006,259.691581,13.168504,Aline Short Entry,-9.833999,19.667999,258.276672,Stock,-1.41,2025-11-02 22:14:23.464595
4,UDR,2.0,35.882137,29.133146,33.689999,33.632474,1.150500,Aline Short Entry,-6.688963,13.377926,33.810000,Stock,-1.35,2025-11-02 22:14:23.464595


## ETF Entries (Aline Entry )

In [60]:
# Fetch the Entry_Signal from buy_list
df3 = df2.copy()
etf_sell = df3[(df3['Type'] == 'ETF') & (df3['Entry Signal'] == 'Aline Short Entry')].reset_index(drop=True)


#etf_sell.to_csv('etf_buy.csv')
etf_sell


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,Type,score,timestamp


## US Stock Entries (Aline Short Entry)

In [61]:
# SP 500 stocks
# Fetch the Entry_Signal from buy_list
sp500_stocks = df2[df2['Type'] == 'Stock'].reset_index(drop=True)


#sp500_stocks.to_csv('etf_buy.csv')
sp500_stocks

,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,Type,score,timestamp
0,CHTR,2.0,254.720382,190.123975,233.839996,233.188246,13.035001,Aline Short Entry,-9.233800,18.467599,228.129995,Stock,-1.62,2025-11-02 22:14:23.464595
1,CMCSA,2.0,29.825525,23.686251,27.840000,27.779100,1.218000,Aline Short Entry,-7.366777,14.733554,27.195550,Stock,-1.44,2025-11-02 22:14:23.464595
2,TMUS,2.0,226.467020,175.774844,210.050003,209.569628,9.607496,Aline Short Entry,-8.062901,16.125802,209.846670,Stock,-1.44,2025-11-02 22:14:23.464595
3,CPAY,2.0,285.229650,208.615444,260.350006,259.691581,13.168504,Aline Short Entry,-9.833999,19.667999,258.276672,Stock,-1.41,2025-11-02 22:14:23.464595
4,UDR,2.0,35.882137,29.133146,33.689999,33.632474,1.150500,Aline Short Entry,-6.688963,13.377926,33.810000,Stock,-1.35,2025-11-02 22:14:23.464595
5,STZ,2.0,138.015497,117.445777,131.380005,131.158923,4.421628,Aline Short Entry,-5.227683,10.455367,130.233337,Stock,-1.23,2025-11-02 22:14:23.464595
6,MAA,2.0,133.932585,116.274017,128.229996,128.046396,3.672004,Aline Short Entry,-4.596919,9.193838,128.245491,Stock,-1.16,2025-11-02 22:14:23.464595
7,WY,2.0,24.342711,20.188353,23.000000,22.957925,0.841500,Aline Short Entry,-6.031843,12.063686,22.986666,Stock,-1.07,2025-11-02 22:14:23.464595
8,CPT,2.0,104.062816,89.859877,99.480003,99.328503,3.030000,Aline Short Entry,-4.766319,9.532637,98.821068,Stock,-1.06,2025-11-02 22:14:23.464595
9,EQR,2.0,63.158652,51.685218,59.439999,59.334174,2.116501,Aline Short Entry,-6.445658,12.891316,59.628157,Stock,-1.04,2025-11-02 22:14:23.464595


In [62]:
# Small Capstocks
# Fetch the Entry_Signal from buy_list
small_cap = df2[df2['Type'] == 'Small'].reset_index(drop=True)


#etf_buy.to_csv('etf_buy.csv')
small_cap

,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Anchored VWAP,Type,score,timestamp
0,BWIN,2.0,24.665110,16.716026,22.100000,22.015415,1.691699,Aline Short Entry,-12.035633,24.071265,21.863000,Small,-1.32,2025-11-02 22:14:23.464595
1,RC,2.0,3.224457,2.317236,2.930000,2.922050,0.159000,Aline Short Entry,-10.349138,20.698277,2.900000,Small,-1.25,2025-11-02 22:14:23.464595
2,LGIH,2.0,46.349282,29.283240,40.810001,40.660601,2.988001,Aline Short Entry,-13.990646,27.981292,40.766668,Small,-1.11,2025-11-02 22:14:23.464595
3,KNF,2.0,64.546475,51.876467,60.459999,60.323139,2.737198,Aline Short Entry,-7.001188,14.002376,59.998333,Small,-1.11,2025-11-02 22:14:23.464595
4,INBK,2.0,20.508738,11.954123,17.740000,17.657200,1.656000,Aline Short Entry,-16.149437,32.298874,17.737657,Small,-0.99,2025-11-02 22:14:23.464595
5,WEST,2.0,5.143552,2.881584,4.410000,4.389562,0.408750,Aline Short Entry,-17.176869,34.353738,4.376667,Small,-0.94,2025-11-02 22:14:23.464595
6,CLW,2.0,19.986722,12.757180,17.650000,17.576875,1.462500,Aline Short Entry,-13.710328,27.420656,17.632442,Small,-0.91,2025-11-02 22:14:23.464595
7,CAL,2.0,12.786605,7.409314,11.040000,10.994175,0.916500,Aline Short Entry,-16.303454,32.606908,10.873333,Small,-0.88,2025-11-02 22:14:23.464595
8,HNST,2.0,3.682599,2.840164,3.410000,3.401788,0.164250,Aline Short Entry,-8.254824,16.509648,3.418123,Small,-0.77,2025-11-02 22:14:23.464595
9,SCL,2.0,46.495348,36.754200,43.349998,43.248298,2.034001,Aline Short Entry,-7.507923,15.015847,43.016666,Small,-0.66,2025-11-02 22:14:23.464595
